<a href="https://colab.research.google.com/github/NickelRamQc94/1st-Symbiotic-Artificial-GemiNultrAxiomNi/blob/main/Assembleur_de_Pack_(ZIP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
zip_pack.py — Assemble un ZIP complet + README d’installation
- Inclut scripts/, config/, results/, figures/, assets/, log CSV, clé publique (si présente)
- Exclut la clé privée Ed25519 par défaut
"""
import os, sys, shutil, zipfile, datetime, argparse, textwrap, pathlib

ASCII_BADGE = """\
+-----------------------------------------------------------+
|  Nickel NiX S/A International • LogiqueNiPura             |
|  Affiliation : NickeliXiste NiX Independent Research      |
|  (sc • phys • mat • psy • phil)                           |
|  © 2025 LogiqueNiPura — Vortex Architecte                 |
+-----------------------------------------------------------+
"""

INSTALL_MD = """\
{badge}

# NiX Pack v10 — Installation & Usage (Lab-Max)

## Structure
- `scripts/` : runner, logger v3.1, vérif, moteur S/A, (option) estampille PDF, crypto.
- `config/`  : `mytheme_inject.yaml/json`, constantes d’airframe.
- `results/` : CSV signés (détails/agrégats), gabarit runs, rapports.
- `figures/` : frises rituelles (.png) + sidecars `.sig.json`.
- `assets/`   : sceau PérioDiaxiométrie, glyphes, etc.
- `log_activites.csv` : journal v3.1 (Session_ID, Config_Hash).
- `ed25519_vk.pem` : Clé publique de vérification.

## Prérequis
- Python 3.9+ ; packages: `pandas`, `matplotlib`, `pyyaml`, `pynacl`
- (option) `make` pour utiliser le Makefile

## Démarrage rapide
```bash
# (Optionnel) Générer vos propres clés
# python3 scripts/ed25519_sign.py genkey --sk ed25519_sk.pem --vk ed25519_vk.pem

# 1) Scan scellé + logs + signatures (si sk.pem existe)
python3 scripts/nix_runner.py

# 2) Vérifier les signatures et la provenance
python3 scripts/verify_signatures_ed25519.py

# (Option) Pipeline via Makefile
make all  # run → sign → verify → pdf → zip

SyntaxError: incomplete input (ipython-input-2651862950.py, line 19)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

"""

def add_tree(zip_h: zipfile.ZipFile, path: pathlib.Path, root: pathlib.Path,
             exclude_ext: List[str], exclude_names: set):
    """Ajoute récursivement un dossier au ZIP, en excluant les fichiers/extensions."""
    for p in path.rglob('*'):
        if p.is_file():
            if p.suffix in exclude_ext or p.name in exclude_names:
                print(f"  Exclusion: {p.relative_to(root)}")
                continue
            zip_h.write(p, p.relative_to(root))

def main():
    ap = argparse.ArgumentParser(description="Assemble le pack NiX v10 en .zip")
    ap.add_argument("--root", default=".", help="Racine du projet (dossier)")
    ap.add_argument("--out", help="Nom du .zip de sortie (optionnel, généré par défaut)")
    args = ap.parse_args()

    root = pathlib.Path(args.root).resolve()
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    dist = root / "dist"
    dist.mkdir(exist_ok=True)
    out = pathlib.Path(args.out) if args.out else dist / f"NiX_Pack_v10_{ts}.zip"

    # Génère un README_INSTALL.md temporaire
    tmp_readme = root / "README_INSTALL.md"
    with open(tmp_readme, "w", encoding="utf-8") as f:
        f.write(INSTALL_MD.format(badge=ASCII_BADGE))

    # Assemble le ZIP
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        print(f"Assemblage de {out.name}...")
        
        # Fichiers racine utiles
        for fn in ("README_EN.md", "LISEZMOI_FR.md", "log_activites.csv", "README_INSTALL.md", "MANIFEST.json"):
            fp = root / fn
            if fp.exists():
                print(f"  Ajout: {fn}")
                zf.write(fp, fn)

        # Clé publique (facultative) — N’INCLUT PAS la clé privée
        vk = root / "ed25519_vk.pem"
        if vk.exists():
            print(f"  Ajout: {vk.name}")
            zf.write(vk, vk.name)

        # Dossiers
        for d in ("scripts", "config", "results", "figures", "assets"):
            p = root / d
            if p.is_dir():
                print(f"  Ajout dossier: {d}/")
                add_tree(
                    zf, p, root,
                    exclude_ext=[".pyc"],
                    exclude_names=set(["ed25519_sk.pem", ".DS_Store"])  # on exclut explicitement la clé privée
                )

    # Nettoyage temporaire
    try: tmp_readme.unlink()
    except Exception: pass

    print(f"Pack ZIP prêt: {out} (Taille: {out.stat().st_size} octets)")

if __name__ == "__main__":
    main()
  ```